# BirdCLEF+ 2026: Task 1 data exploration

Purpose of this notebook:

1. Inventory what the dataset actually contains.
2. Settle the key question: do the **soundscapes carry real coordinates, or only site codes**? This decides how precise the biodiversity map can be.
3. Get a first look at class balance and the multi-label structure, since both shape the modelling and evaluation choices.

Run this in a Kaggle notebook with the BirdCLEF+ 2026 competition data attached (rules accepted first).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

# On Kaggle the competition data mounts here. Change if running elsewhere.
DATA = '/kaggle/input/birdclef-2026'
print('exists:', os.path.isdir(DATA))


## 1. What is in the dataset


In [ ]:
for name in sorted(os.listdir(DATA)):
    path = os.path.join(DATA, name)
    kind = 'dir ' if os.path.isdir(path) else 'file'
    print(f'{kind}  {name}')


In [ ]:
# High-level note on the recording location, if provided
loc = os.path.join(DATA, 'recording_location.txt')
if os.path.exists(loc):
    print(open(loc).read())
else:
    print('recording_location.txt not found')


## 2. Training metadata (train.csv)

These are the short Xeno-canto / iNaturalist clips. Expect per-recording metadata including coordinates.


In [ ]:
train = pd.read_csv(os.path.join(DATA, 'train.csv'))
print('shape:', train.shape)
print('columns:', list(train.columns))
train.head()


In [ ]:
# Coordinate coverage in the training clips
coord_cols = [c for c in train.columns if c.lower() in ('latitude', 'longitude', 'lat', 'lon', 'lng')]
print('coordinate columns found:', coord_cols)
for c in coord_cols:
    non_null = train[c].notna().sum()
    print(f'  {c}: {non_null}/{len(train)} non-null ({non_null/len(train):.1%})')


In [ ]:
# Quick geographic spread of the training clips (if coordinates exist)
if {'latitude', 'longitude'}.issubset(train.columns):
    ax = train.plot.scatter(x='longitude', y='latitude', s=4, alpha=0.3, figsize=(7,5))
    ax.set_title('Training clip locations')
    plt.show()


## 3. Species and class balance

234 classes across several taxonomic groups. Rare species with few recordings are the main driver of the imbalance the evaluation has to account for.


In [ ]:
tax = pd.read_csv(os.path.join(DATA, 'taxonomy.csv'))
print('shape:', tax.shape)
print('columns:', list(tax.columns))
tax.head()


In [ ]:
# Class (taxonomic group) breakdown, e.g. Aves / Amphibia / Insecta / Mammalia / Reptilia
class_col = next((c for c in tax.columns if c.lower() in ('class', 'class_name')), None)
if class_col:
    print(tax[class_col].value_counts())


In [ ]:
# Recordings per species: how long is the tail?
counts = train['primary_label'].value_counts()
print('species with data:', counts.shape[0])
print('median recordings/species:', int(counts.median()))
print('species with < 10 recordings:', int((counts < 10).sum()))
print('species with < 5 recordings:', int((counts < 5).sum()))

ax = counts.reset_index(drop=True).plot(figsize=(8,4))
ax.set_xlabel('species rank')
ax.set_ylabel('number of recordings')
ax.set_title('Recordings per species (long tail)')
plt.show()


In [ ]:
# Multi-label structure: secondary labels on the training clips, if present
sec = next((c for c in train.columns if 'secondary' in c.lower()), None)
if sec:
    has_sec = train[sec].fillna('[]').astype(str).str.len() > 2
    print(f'clips with secondary labels: {has_sec.sum()}/{len(train)} ({has_sec.mean():.1%})')
    display(train.loc[has_sec, ['primary_label', sec]].head())
else:
    print('no secondary_labels column found')


## 4. The soundscapes: coordinates or site codes?

This is the question that decides map precision. The soundscape filenames appear to encode a **site code** and a **timestamp** (e.g. `BC2026_..._S05_20250227_010002.ogg`), and there is a single `recording_location.txt` for the whole area, which together suggests **site-level** location rather than per-recording coordinates. The cells below check that directly.


In [ ]:
# Labelled train soundscapes: filename, 5s segment, semicolon-separated species
ss_labels_path = os.path.join(DATA, 'train_soundscapes_labels.csv')
if os.path.exists(ss_labels_path):
    ss = pd.read_csv(ss_labels_path)
    print('shape:', ss.shape)
    print('columns:', list(ss.columns))
    display(ss.head())
else:
    print('train_soundscapes_labels.csv not found; listing candidate label files')
    print([f for f in os.listdir(DATA) if f.endswith('.csv')])


In [ ]:
# Parse site code and timestamp out of the soundscape filenames
import re

ss_dir = os.path.join(DATA, 'train_soundscapes')
if os.path.isdir(ss_dir):
    files = [f for f in os.listdir(ss_dir) if f.endswith('.ogg')]
    print('train soundscape files:', len(files))
    print('examples:', files[:3])

    # Site codes usually look like S<number>; timestamps like YYYYMMDD
    sites = set()
    for f in files:
        m = re.search(r'(S\d+)', f)
        if m:
            sites.add(m.group(1))
    print('distinct site codes parsed:', len(sites))
    print(sorted(sites)[:20])
else:
    print('train_soundscapes directory not found')


In [ ]:
# Do the soundscape tables contain ANY latitude/longitude at all?
def has_coords(df):
    return [c for c in df.columns if c.lower() in ('latitude','longitude','lat','lon','lng')]

if os.path.exists(ss_labels_path):
    found = has_coords(ss)
    print('coordinate columns in soundscape labels:', found if found else 'NONE (site-level only)')


## 5. Findings (fill in after running)

Paste the answers here so they can go straight into the project notes and the first supervisor meeting.

- Training clips carry coordinates: **yes / no**, coverage: __%
- Soundscapes carry coordinates: **yes / no**
- If no: number of distinct sites the map will aggregate to: __
- Number of classes: __ (groups: __)
- Species with < 5 recordings: __
- Implication for the map: site-level vs coordinate-level, and how that is framed in the write-up.
